#1. Thiết lập môi trường và tiên xử lý index Datetime
Khởi tạo đường dẫn dự án Project/data/raw/beijing/ để đồng bộ với cấu trúc của nhánh GitHub. Nạp tệp dữ liệu thô và chuyển đổi cột thời gian thành DatetimeIndex.

In [19]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf
import warnings
warnings.filterwarnings('ignore')

print("=== BẮT ĐẦU XỬ LÝ P0 CHO BẮC KINH ===")

# 1. SETUP PATH & FOLDERS
RAW_DIR = "Project/data/raw/beijing/"
OUT_DIR = "Project/notebook/"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)


file_path = f"{RAW_DIR}PRSA_data_2010.1.1-2014.12.31.csv"
if not os.path.exists(file_path):
    print("Vui lòng đưa file PRSA_data_2010.1.1-2014.12.31.csv vào đúng thư mục:", RAW_DIR)
else:
    df = pd.read_csv(file_path)

    # Tiền xử lý index Datetime
    df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
    df.set_index('datetime', inplace=True)
    df.drop(columns=['No', 'year', 'month', 'day', 'hour'], inplace=True, errors='ignore')

    # Bỏ dòng NaN ở những giờ đầu tiên (do máy đo chưa khởi động) để không ảnh hưởng audit
    first_valid = df['pm2.5'].first_valid_index()
    df = df.loc[first_valid:]

=== BẮT ĐẦU XỬ LÝ P0 CHO BẮC KINH ===


#2. Data Audit (Kiểm toán Dữ liệu Thô)
Thực hiện quét toàn diện dữ liệu: số dòng, trùng lặp, phần trăm missing, tháng bị thiếu nhiều nhất và khoảng trống liên tiếp lớn nhất. Tạo cờ đánh dấu chất lượng (quality flags) và xuất báo cáo beijing_data_quality_report.csv.

In [20]:
    audit_data = []
    audit_data.append({'Metric': 'Total Rows', 'Value': len(df)})
    audit_data.append({'Metric': 'Duplicate Rows', 'Value': df.duplicated().sum()})
    audit_data.append({'Metric': 'Missing PM2.5 (%)', 'Value': f"{(df['pm2.5'].isnull().sum() / len(df)) * 100:.2f}%"})

    missing_by_ym = df['pm2.5'].isnull().groupby([df.index.year, df.index.month]).sum()
    max_missing_month = missing_by_ym.idxmax()
    audit_data.append({'Metric': 'Max Missing Month (Year, Month)', 'Value': f"{max_missing_month} ({missing_by_ym.max()} hours)"})

    gaps = df['pm2.5'].isnull().astype(int).groupby(df['pm2.5'].notnull().astype(int).cumsum()).sum()
    audit_data.append({'Metric': 'Max Consecutive Gap (hours)', 'Value': gaps.max()})

    # Tạo Quality Flags
    df['pm2.5_missing_flag'] = df['pm2.5'].isnull().astype(int)

    # LƯU OUTPUT 1: beijing_data_quality_report.csv
    pd.DataFrame(audit_data).to_csv(f"{OUT_DIR}beijing_data_quality_report.csv", index=False)
    print("✓ Đã xong Data Audit.")

✓ Đã xong Data Audit.


#3. Làm sạch Dữ liệu
Sử dụng nội suy tuyến tính một chiều (forward fill) với giới hạn 24 giờ. Tuyệt đối không dùng nội suy hai phía (both) để tránh việc mô hình học máy lấy thông tin của tương lai điền cho quá khứ. Mã hóa One-Hot biến hướng gió.

In [21]:
    df['pm2.5_clean'] = df['pm2.5'].interpolate(method='linear', limit_direction='forward', limit=24)
    df = pd.get_dummies(df, columns=['cbwd'], prefix='wind', dtype=int)

#4. Kỹ thuật Trích xuất Đặc trưng  & Chia tập

Tập ML (Hourly): Tạo Lags (1, 2, 3, 6, 12, 24, 48, 72h) và Rolling (Mean 3-24h, Std 24h, Max 24h).

Tập Health GAM (Daily): Tổng hợp dữ liệu theo ngày và trích xuất Lag từ 0 đến 7 ngày.

Phân chia: Train (2010-2012) | Validation (2013) | Test (2014).

In [22]:
    df_ml = df.copy()

    # A. ML Lags (Hourly: 1, 2, 3, 6, 12, 24, 48, 72)
    lags_ml = [1, 2, 3, 6, 12, 24, 48, 72]
    for lag in lags_ml:
        df_ml[f'pm2.5_lag_{lag}h'] = df_ml['pm2.5_clean'].shift(lag)

    # B. Bổ sung Rolling (Mean: 3, 6, 12, 24 | Std/Max: 24)
    for w in [3, 6, 12, 24]:
        df_ml[f'pm2.5_rolling_mean_{w}h'] = df_ml['pm2.5_clean'].shift(1).rolling(window=w).mean()
    df_ml['pm2.5_rolling_std_24h'] = df_ml['pm2.5_clean'].shift(1).rolling(window=24).std()
    df_ml['pm2.5_rolling_max_24h'] = df_ml['pm2.5_clean'].shift(1).rolling(window=24).max()

    # Mục tiêu dự báo (Target) & Nhãn năm
    df_ml['target_t1'] = df_ml['pm2.5_clean'].shift(-1)
    df_ml['target_t24'] = df_ml['pm2.5_clean'].shift(-24)
    df_ml['year'] = df_ml.index.year

    # LƯU OUTPUT 2: beijing_hourly_clean.csv
    df_ml.drop(columns=['year'], errors='ignore').to_csv(f"{OUT_DIR}beijing_hourly_clean.csv")
    print("✓ Đã xong đặc trưng ML Hourly.")

    # C. Health Lags (Daily: Lag0 -> Lag7)
    agg_rules = {'pm2.5_clean': 'mean', 'DEWP': 'mean', 'TEMP': 'mean', 'PRES': 'mean', 'Iws': 'mean', 'Is': 'sum', 'Ir': 'sum', 'wind_NE': 'sum', 'wind_NW': 'sum', 'wind_SE': 'sum', 'wind_cv': 'sum'}
    df_gam = df_ml.resample('D').agg(agg_rules)

    for i in range(1, 8):
        df_gam[f'pm2.5_lag_{i}d'] = df_gam['pm2.5_clean'].shift(i)



    # CHIA TẬP KHẮT KHE (TRAIN 2010-2012 | VAL 2013 | TEST 2014)

    # Phải xóa NaN sinh ra do trễ 72h thì data matrix mới "sạch" hoàn toàn
    df_model = df_ml.dropna(subset=[f'pm2.5_lag_{lags_ml[-1]}h', 'target_t24'])

    train = df_model[df_model['year'] <= 2012].drop(columns=['year'])
    val = df_model[df_model['year'] == 2013].drop(columns=['year'])
    test = df_model[df_model['year'] == 2014].drop(columns=['year'])

    # LƯU OUTPUT 3, 4, 5: beijing_train.csv, validation, test
    train.to_csv(f"{OUT_DIR}beijing_train.csv")
    val.to_csv(f"{OUT_DIR}beijing_validation.csv")
    test.to_csv(f"{OUT_DIR}beijing_test.csv")

    # LƯU OUTPUT 6: feature_dictionary.csv
    pd.DataFrame({'Feature': train.columns, 'Type': train.dtypes.astype(str)}).to_csv(f"{OUT_DIR}feature_dictionary.csv", index=False)
    print("✓ Đã xong quy trình Split (Train/Val/Test).")

✓ Đã xong đặc trưng ML Hourly.
✓ Đã xong quy trình Split (Train/Val/Test).


#5. Phân tích Khám phá Dữ liệu (EDA)Trực quan hóa dữ liệu toàn diện bao gồm:






##5.1. Xu hướng chuỗi thời gian & Sự kiện cực đoan (Time Series & Extreme PM2.5)

In [23]:
    plt.style.use('seaborn-v0_8-whitegrid')

    # A. Time Series & Extreme PM2.5
    plt.figure(figsize=(15, 5))
    plt.plot(df.index, df['pm2.5_clean'], color='black', alpha=0.5, linewidth=0.5)
    plt.axhline(df['pm2.5_clean'].quantile(0.99), color='red', linestyle='--', label='99th Percentile')
    plt.title('Time Series of PM2.5 with Extreme Threshold')
    plt.legend()
    plt.savefig(f"{OUT_DIR}EDA_1_TimeSeries.png", bbox_inches='tight')
    plt.close()


##5.2. Tính mùa vụ (Monthly).

In [24]:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df.index.month, y=df['pm2.5_clean'], palette='coolwarm')
    plt.title('Monthly PM2.5 Distribution (Seasonality)')
    plt.savefig(f"{OUT_DIR}EDA_3_Monthly.png", bbox_inches='tight')
    plt.close()

##5.3. Tính nhật triều trong ngày (Hourly).

In [25]:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df.index.hour, y=df['pm2.5_clean'], palette='viridis')
    plt.title('Hourly PM2.5 Distribution (Diurnal)')
    plt.savefig(f"{OUT_DIR}EDA_2_Hourly.png", bbox_inches='tight')
    plt.close()

##5.4. Ma trận tương quan (Correlation).

In [26]:
    plt.figure(figsize=(8, 6))
    sns.heatmap(df[['pm2.5_clean', 'DEWP', 'TEMP', 'PRES', 'Iws', 'Is', 'Ir']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Matrix')
    plt.savefig(f"{OUT_DIR}EDA_4_Correlation.png", bbox_inches='tight')
    plt.close()

##5.5. Quan hệ giữa nhiệt độ, tốc độ gió và  𝑃𝑀2.5 .

In [27]:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x='TEMP', y='pm2.5_clean', hue='Iws', data=df.sample(2000, random_state=42), palette='magma', alpha=0.6)
    plt.title('Weather Relationship: Temp vs PM2.5 (Color: Wind)')
    plt.savefig(f"{OUT_DIR}EDA_5_Weather.png", bbox_inches='tight')
    plt.close()

##5.6. Hàm tự tương quan ACF (72 Lags).

In [28]:
    fig, ax = plt.subplots(figsize=(12, 4))
    plot_acf(df['pm2.5_clean'].dropna(), lags=72, ax=ax, title='ACF - PM2.5 (72 Lags)')
    plt.savefig(f"{OUT_DIR}EDA_6_ACF.png", bbox_inches='tight')
    plt.close()

In [29]:
print(f"✓ Đã lưu trọn bộ biểu đồ EDA vào: {OUT_DIR}")


✓ Đã lưu trọn bộ biểu đồ EDA vào: Project/notebook/
